In [1]:
import sys

print(sys.executable)

/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/bin/python


In [9]:
import numpy as np

In [3]:
uv add numpy

/home/keerthivardhan/Desktop/ProductionProjects/autonomus-assistent/.venv/bin/python: No module named uv
Note: you may need to restart the kernel to use updated packages.


In [58]:
sentence = "cat eats fish"
tokens = sentence.lower().split()
tokens

['cat', 'eats', 'fish']

In [16]:
vocab = {
    "cat": 0,
    "eats": 1,
    "fish": 2,
    "<unk>": 3,
    "drinks":4,
    "milk":5
}

print(vocab)

{'cat': 0, 'eats': 1, 'fish': 2, '<unk>': 3, 'drinks': 4, 'milk': 5}


In [7]:
tokens_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
tokens_ids

[0, 1, 2]

In [10]:
embedding_dim = 4
vocab_size = len(vocab)

np.random.seed(42)

embedding_table = np.random.randn(vocab_size, embedding_dim)

embedding_table

array([[ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986],
       [-0.23415337, -0.23413696,  1.57921282,  0.76743473],
       [-0.46947439,  0.54256004, -0.46341769, -0.46572975],
       [ 0.24196227, -1.91328024, -1.72491783, -0.56228753]])

In [14]:
embeddings = embedding_table[tokens_ids]
embeddings

array([[ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986],
       [-0.23415337, -0.23413696,  1.57921282,  0.76743473],
       [-0.46947439,  0.54256004, -0.46341769, -0.46572975]])

In [15]:
for word, idx in vocab.items():
    print(f"{word:6} -> {embedding_table[idx]}")

cat    -> [ 0.49671415 -0.1382643   0.64768854  1.52302986]
eats   -> [-0.23415337 -0.23413696  1.57921282  0.76743473]
fish   -> [-0.46947439  0.54256004 -0.46341769 -0.46572975]
<unk>  -> [ 0.24196227 -1.91328024 -1.72491783 -0.56228753]


In [17]:
sentence2 = "cat drinks milk"

In [18]:
tokens = sentence2.lower().split()
token_ids2 =[vocab.get(token, vocab["<unk>"]) for token in tokens ]
token_ids2

[0, 4, 5]

In [20]:
embedding_dim = 6
vocab_size = len(vocab)

np.random.seed(42)

embedding_table2 = np.random.randn(vocab_size, embedding_dim)

embedding_table2

array([[ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986, -0.23415337,
        -0.23413696],
       [ 1.57921282,  0.76743473, -0.46947439,  0.54256004, -0.46341769,
        -0.46572975],
       [ 0.24196227, -1.91328024, -1.72491783, -0.56228753, -1.01283112,
         0.31424733],
       [-0.90802408, -1.4123037 ,  1.46564877, -0.2257763 ,  0.0675282 ,
        -1.42474819],
       [-0.54438272,  0.11092259, -1.15099358,  0.37569802, -0.60063869,
        -0.29169375],
       [-0.60170661,  1.85227818, -0.01349722, -1.05771093,  0.82254491,
        -1.22084365]])

In [22]:
embeddings2 = embedding_table2[token_ids2]
embeddings2

array([[ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986, -0.23415337,
        -0.23413696],
       [-0.54438272,  0.11092259, -1.15099358,  0.37569802, -0.60063869,
        -0.29169375],
       [-0.60170661,  1.85227818, -0.01349722, -1.05771093,  0.82254491,
        -1.22084365]])

## Lab 2: Positional Encoding.

We'll build Positional Encoding from scratch and visualize why "cat eats fish" is different from "fish eats cat" even though they contain the same words.
The problem is that an embedding tells us what the token is, but not where it occurs.

In [24]:
import numpy as np

tokens = ["cat", "eats", "fish"]

vocab = {
    "cat": 0,
    "eats": 1,
    "fish": 2
}

token_ids = [vocab[token] for token in tokens]

np.random.seed(42)

embedding_dim = 4
embedding_table = np.random.randn(len(vocab), embedding_dim)

embeddings = embedding_table[token_ids]

print(embeddings)
print(embeddings.shape)

[[ 0.49671415 -0.1382643   0.64768854  1.52302986]
 [-0.23415337 -0.23413696  1.57921282  0.76743473]
 [-0.46947439  0.54256004 -0.46341769 -0.46572975]]
(3, 4)


In [25]:
# positional information

In [26]:
positions = np.array([
    [0, 0, 0, 0],
    [1, 1, 1, 1],
    [2, 2, 2, 2]
])

print(positions)

[[0 0 0 0]
 [1 1 1 1]
 [2 2 2 2]]


cat   → position 0
eats  → position 1
fish  → position 2

In [27]:
combined = embeddings + positions

print(combined)

[[ 0.49671415 -0.1382643   0.64768854  1.52302986]
 [ 0.76584663  0.76586304  2.57921282  1.76743473]
 [ 1.53052561  2.54256004  1.53658231  1.53427025]]


Now the representation contains:

"What token am I?" + "Where am I?"

### Lab 2.5 — Sinusoidal Positional Encoding

This is the important experiment.

The formulas are:

$$ PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d}}\right) $$ $$ PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d}}\right) $$

Don't worry about memorizing these formulas yet.

Let's see them first.

In [28]:
def positional_encoding(seq_len, embedding_dim):
    pe = np.zeros((seq_len, embedding_dim))

    for pos in range(seq_len):
        for i in range(0, embedding_dim, 2):
            pe[pos, i] = np.sin(
                pos / (10000 ** (i / embedding_dim))
            )

            pe[pos, i + 1] = np.cos(
                pos / (10000 ** (i / embedding_dim))
            )

    return pe

In [29]:
pe = positional_encoding(
    seq_len=3,
    embedding_dim=4
)

print(pe)

[[ 0.          1.          0.          1.        ]
 [ 0.84147098  0.54030231  0.00999983  0.99995   ]
 [ 0.90929743 -0.41614684  0.01999867  0.99980001]]


In [30]:
encoded = embeddings + pe

print(encoded)

[[ 0.49671415  0.8617357   0.64768854  2.52302986]
 [ 0.60731761  0.30616535  1.58921265  1.76738473]
 [ 0.43982304  0.12641321 -0.44341903  0.53407025]]


## Lab 3 — Self-Attention From Scratch

In [32]:
import numpy as np

X = np.array([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 1.0],
    [1.0, 1.0, 0.0]
])

print(X)
print(X.shape)

[[1. 0. 1.]
 [0. 1. 1.]
 [1. 1. 0.]]
(3, 3)


In [33]:
scores = X @ X.T

print(scores)

[[2. 1. 1.]
 [1. 2. 1.]
 [1. 1. 2.]]


In [34]:
def softmax(x):
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [35]:
attention_weights = softmax(scores)

print(attention_weights)

[[0.57611688 0.21194156 0.21194156]
 [0.21194156 0.57611688 0.21194156]
 [0.21194156 0.21194156 0.57611688]]


In [36]:
output = attention_weights @ X

print(output)

[[0.78805844 0.42388312 0.78805844]
 [0.42388312 0.78805844 0.78805844]
 [0.78805844 0.78805844 0.42388312]]


In [37]:
import numpy as np

X = np.array([
    [1.0, 0.0, 1.0, 0.0],  # cat
    [0.0, 1.0, 1.0, 0.0],  # eats
    [1.0, 1.0, 0.0, 1.0]   # fish
])

print(X)
print("Shape:", X.shape)

[[1. 0. 1. 0.]
 [0. 1. 1. 0.]
 [1. 1. 0. 1.]]
Shape: (3, 4)


In [38]:
np.random.seed(42)

WQ = np.random.randn(4, 2)
WK = np.random.randn(4, 2)
WV = np.random.randn(4, 2)

In [39]:
Q = X @ WQ
K = X @ WK
V = X @ WV

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: (3, 2)
K shape: (3, 2)
V shape: (3, 2)


In [40]:
scores = Q @ K.T

print(scores)
print(scores.shape)

[[ 0.45072217  0.82780081 -0.51705175]
 [-1.86079579 -3.1578687  -1.72480024]
 [-3.56971993 -5.72326541 -8.2836535 ]]
(3, 3)


In [41]:
d_k = Q.shape[-1]

scaled_scores = scores / np.sqrt(d_k)

print(scaled_scores)

[[ 0.3187087   0.58534357 -0.3656108 ]
 [-1.31578132 -2.23295037 -1.21961795]
 [-2.52417317 -4.04695978 -5.85742756]]


In [42]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

In [43]:
attention_weights = softmax(scaled_scores)

print(attention_weights)

[[0.35587226 0.46461389 0.17951385]
 [0.39990598 0.15982196 0.44027206]
 [0.79758804 0.17395656 0.0284554 ]]


In [44]:
output = attention_weights @ V

print(output)
print(output.shape)

[[ 0.08752756 -1.18246869]
 [-0.54576293 -1.33714141]
 [ 0.40542726 -0.28617873]]
(3, 2)


In [45]:
WQ

array([[ 0.49671415, -0.1382643 ],
       [ 0.64768854,  1.52302986],
       [-0.23415337, -0.23413696],
       [ 1.57921282,  0.76743473]])

In [46]:
print("X:\n", X)
print("\nQ:\n", Q)
print("\nK:\n", K)
print("\nV:\n", V)
print("\nAttention weights:\n", attention_weights)
print("\nOutput:\n", output)

X:
 [[1. 0. 1. 0.]
 [0. 1. 1. 0.]
 [1. 1. 0. 1.]]

Q:
 [[ 0.26256078 -0.37240126]
 [ 0.41353516  1.2888929 ]
 [ 2.72361551  2.15220028]]

K:
 [[-0.22751211 -1.3707202 ]
 [-0.22145542 -2.37901   ]
 [-2.65780991 -0.48545724]]

V:
 [[ 0.45281765  0.08847103]
 [ 0.55762469 -1.63808   ]
 [-1.85332699 -2.52280455]]

Attention weights:
 [[0.35587226 0.46461389 0.17951385]
 [0.39990598 0.15982196 0.44027206]
 [0.79758804 0.17395656 0.0284554 ]]

Output:
 [[ 0.08752756 -1.18246869]
 [-0.54576293 -1.33714141]
 [ 0.40542726 -0.28617873]]


## Lab 6 — Multi-Head Attention From Scratch

In [48]:
d_model = 4
num_heads = 2
head_dim = 2
import numpy as np

X = np.array([
    [1.0, 0.0, 1.0, 0.0],  # cat
    [0.0, 1.0, 1.0, 0.0],  # eats
    [1.0, 1.0, 0.0, 1.0]   # fish
])

print(X)
print("Shape:", X.shape)

[[1. 0. 1. 0.]
 [0. 1. 1. 0.]
 [1. 1. 0. 1.]]
Shape: (3, 4)


In [49]:
np.random.seed(42)

d_model = 4
num_heads = 2
head_dim = 2

WQ = [
    np.random.randn(d_model, head_dim),
    np.random.randn(d_model, head_dim)
]

WK = [
    np.random.randn(d_model, head_dim),
    np.random.randn(d_model, head_dim)
]

WV = [
    np.random.randn(d_model, head_dim),
    np.random.randn(d_model, head_dim)
]

In [50]:
def attention_head(X, WQ, WK, WV):
    Q = X @ WQ
    K = X @ WK
    V = X @ WV

    scores = Q @ K.T
    scores = scores / np.sqrt(K.shape[-1])

    weights = softmax(scores)

    output = weights @ V

    return output, weights

In [51]:
head1_output, head1_weights = attention_head(
    X,
    WQ[0],
    WK[0],
    WV[0]
)

print("Head 1 output:")
print(head1_output)

print("\nHead 1 attention weights:")
print(head1_weights)

Head 1 output:
[[ 0.3022123  -2.77376529]
 [ 0.3103747  -3.01004098]
 [ 0.26312335 -3.03044016]]

Head 1 attention weights:
[[2.56225320e-01 4.11646028e-01 3.32128652e-01]
 [7.93072756e-01 1.69526525e-01 3.74007192e-02]
 [9.18577791e-01 8.12187887e-02 2.03420698e-04]]


In [52]:
head2_output, head2_weights = attention_head(
    X,
    WQ[1],
    WK[1],
    WV[1]
)

print("Head 2 output:")
print(head2_output)

print("\nHead 2 attention weights:")
print(head2_weights)

Head 2 output:
[[-1.06395487 -0.67541702]
 [-1.08044668 -0.7269672 ]
 [-0.62410405 -0.00796606]]

Head 2 attention weights:
[[0.5108192  0.43570789 0.05347291]
 [0.58083032 0.40913308 0.0100366 ]
 [0.13288562 0.37941731 0.48769706]]


In [53]:
print("Head 1:")
print(head1_weights)

print("\nHead 2:")
print(head2_weights)

Head 1:
[[2.56225320e-01 4.11646028e-01 3.32128652e-01]
 [7.93072756e-01 1.69526525e-01 3.74007192e-02]
 [9.18577791e-01 8.12187887e-02 2.03420698e-04]]

Head 2:
[[0.5108192  0.43570789 0.05347291]
 [0.58083032 0.40913308 0.0100366 ]
 [0.13288562 0.37941731 0.48769706]]


In [54]:
multi_head_output = np.concatenate(
    [head1_output, head2_output],
    axis=-1
)

print(multi_head_output)
print("Shape:", multi_head_output.shape)

[[ 0.3022123  -2.77376529 -1.06395487 -0.67541702]
 [ 0.3103747  -3.01004098 -1.08044668 -0.7269672 ]
 [ 0.26312335 -3.03044016 -0.62410405 -0.00796606]]
Shape: (3, 4)


In [55]:
WO = np.random.randn(
    num_heads * head_dim,
    d_model
)

output = multi_head_output @ WO

print(output)
print("Shape:", output.shape)

[[ 3.19800043 -1.77507364 -2.3670204  -2.92952675]
 [ 3.39928713 -1.91931828 -2.55640659 -3.10713264]
 [ 2.66936179 -2.12308576 -3.23703802 -3.52282543]]
Shape: (3, 4)


In [77]:
def getEmbeddings(sentance, embedding_table, vocab):
    tokens = sentance.lower().split()
    tokens_ids = [vocab.get(token, vocab["<unk>"]) for token in tokens]
    embeddings = embedding_table[tokens_ids]
    return embeddings

def addPositionalInfomation(embeddings , position_info):
    positional_embeddings = embeddings + position_info
    return positional_embeddings

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)


def buildSimpleAttentions(embeddings):
    attention_weights= embeddings@embeddings.T
    softed_attention_weights = softmax(attention_weights)
    output = softed_attention_weights @ embeddings
    return output

def encoder(sentence):

    vocab = {
        "cat": 0,
        "eats": 1,
        "fish": 2,
        "<unk>": 3,
        "drinks":4,
        "milk":5
    }
    
    embedding_dim = 4
    vocab_size = len(vocab)
    
    np.random.seed(42)
    embedding_table = np.random.randn(vocab_size, embedding_dim)

    

    positional_info = [
        [1,0,0,0],
        [0,1,0,0],
        [0,0,1,0]
    ]

    embeddings = getEmbeddings(sentence, embedding_table, vocab)
    positional_embeddings= addPositionalInfomation(positional_info , embeddings)
    encodings = buildSimpleAttentions(positional_embeddings)

    return encodings

In [78]:

sentence = "cat eats fish"
encoding = encoder(sentence)

In [79]:
encoding

array([[ 1.42896616, -0.10359443,  0.68172241,  1.49110869],
       [-0.05036711,  0.65069683,  1.41557916,  0.78509196],
       [-0.25097882,  0.61066218,  1.03443797,  0.22465598]])